# Статистическая проверка гипотез: продажи супермаркета

Ноутбук дополняет Power BI-дашборд статистическим анализом. Проверяются различия среднего чека и количества товаров между Member и Normal, а также связь типа клиента со способом оплаты.

**Важно:** Member и Normal — наблюдаемые группы, а не случайно назначенные экспериментальные группы. Поэтому это статистический анализ наблюдательных данных, не настоящий причинный A/B-тест.

Уровень значимости: 5%. Для трёх проверок используется поправка Holm на множественные сравнения. Для количественных метрик применяются t-тест Уэлча, 95%-й доверительный интервал и Cohen's d. Для категориальных переменных — χ²-тест независимости и V Крамера.

Источник набора данных: selva86/datasets, файл supermarket_sales.csv; исходный набор содержит 1 000 транзакций.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

# Load the same 1,000-row supermarket dataset used for the dashboard.
local_candidates = [
    Path("../data/supermarket_sales.csv"),
    Path("data/supermarket_sales.csv"),
]
data_path = next((p for p in local_candidates if p.exists()), None)
if data_path is not None:
    df = pd.read_csv(data_path)
else:
    url = "https://raw.githubusercontent.com/bignnwp/ML_Analytics_Portfolio_Pavlov/main/data/supermarket_sales.csv"
    df = pd.read_csv(url)

required = {"Customer type", "Total", "Quantity", "Payment"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")
df["Total"] = pd.to_numeric(df["Total"], errors="raise")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="raise")
assert len(df) == 1000, f"Expected 1,000 records, found {len(df)}"

ALPHA = 0.05

def welch_test(metric, hypothesis, test_id):
    member = df.loc[df["Customer type"].eq("Member"), metric].dropna().astype(float)
    normal = df.loc[df["Customer type"].eq("Normal"), metric].dropna().astype(float)

    result = stats.ttest_ind(member, normal, equal_var=False, alternative="two-sided")
    mean_member, mean_normal = member.mean(), normal.mean()
    diff = mean_member - mean_normal
    se = np.sqrt(member.var(ddof=1) / len(member) + normal.var(ddof=1) / len(normal))
    var_a, var_b = member.var(ddof=1), normal.var(ddof=1)
    df_welch = (var_a / len(member) + var_b / len(normal)) ** 2 / (
        (var_a / len(member)) ** 2 / (len(member) - 1)
        + (var_b / len(normal)) ** 2 / (len(normal) - 1)
    )
    critical = stats.t.ppf(1 - ALPHA / 2, df_welch)
    pooled_sd = np.sqrt(
        ((len(member) - 1) * var_a + (len(normal) - 1) * var_b)
        / (len(member) + len(normal) - 2)
    )
    return {
        "Test_ID": test_id,
        "Hypothesis": hypothesis,
        "Metric": metric,
        "Test": "Welch t-test",
        "Group_A": "Member",
        "Group_B": "Normal",
        "N_A": len(member),
        "N_B": len(normal),
        "Mean_A": mean_member,
        "Mean_B": mean_normal,
        "Difference_A_minus_B": diff,
        "Difference_pct_vs_B": diff / mean_normal * 100,
        "CI95_Lower": diff - critical * se,
        "CI95_Upper": diff + critical * se,
        "Test_Statistic": result.statistic,
        "Degrees_of_Freedom": df_welch,
        "P_Value": result.pvalue,
        "Effect_Size": diff / pooled_sd,
        "Effect_Size_Name": "Cohen's d",
        "Notes": "Observational comparison; no causal effect is established.",
    }

results = [
    welch_test("Total", "H1: средний чек Member и Normal различается", "H1"),
    welch_test("Quantity", "H2: число товаров в транзакции Member и Normal различается", "H2"),
]

# H3: association between customer type and payment method.
contingency = pd.crosstab(df["Customer type"], df["Payment"]).reindex(
    index=["Member", "Normal"], columns=["Cash", "Credit card", "Ewallet"], fill_value=0
)
chi2, p_value, dof, expected = stats.chi2_contingency(contingency, correction=False)
n = contingency.to_numpy().sum()
cramers_v = np.sqrt(chi2 / (n * min(contingency.shape[0] - 1, contingency.shape[1] - 1)))
results.append({
    "Test_ID": "H3",
    "Hypothesis": "H3: распределение способов оплаты связано с типом клиента",
    "Metric": "Payment × Customer type",
    "Test": "Pearson chi-square",
    "Group_A": "Member",
    "Group_B": "Normal",
    "N_A": int(contingency.loc["Member"].sum()),
    "N_B": int(contingency.loc["Normal"].sum()),
    "Mean_A": np.nan, "Mean_B": np.nan, "Difference_A_minus_B": np.nan,
    "Difference_pct_vs_B": np.nan, "CI95_Lower": np.nan, "CI95_Upper": np.nan,
    "Test_Statistic": chi2, "Degrees_of_Freedom": dof, "P_Value": p_value,
    "Effect_Size": cramers_v, "Effect_Size_Name": "Cramer's V",
    "Notes": "Association test; not a causal conclusion.",
})

results_df = pd.DataFrame(results)
results_df["P_Value_Holm"] = multipletests(results_df["P_Value"], alpha=ALPHA, method="holm")[1]
results_df["Alpha"] = ALPHA
results_df["Significant_After_Holm"] = results_df["P_Value_Holm"] < ALPHA
results_df["Conclusion"] = np.where(
    results_df["Significant_After_Holm"],
    "Статистически значимое различие/связь после поправки Holm",
    "Нет статистически значимого различия/связи после поправки Holm",
)

# Use stable column order for the CSV that is loaded into Power BI.
columns = [
    "Test_ID", "Hypothesis", "Metric", "Test", "Group_A", "Group_B", "N_A", "N_B",
    "Mean_A", "Mean_B", "Difference_A_minus_B", "Difference_pct_vs_B",
    "CI95_Lower", "CI95_Upper", "Test_Statistic", "Degrees_of_Freedom",
    "P_Value", "P_Value_Holm", "Effect_Size", "Effect_Size_Name",
    "Alpha", "Significant_After_Holm", "Conclusion", "Notes"
]
results_df = results_df[columns]
results_df.to_csv("hypothesis_test_results.csv", index=False, encoding="utf-8-sig")

payment_distribution = (
    contingency.rename_axis(index="Customer type", columns="Payment")
    .stack()
    .rename("Transactions")
    .reset_index()
)
payment_distribution["Customer type total"] = payment_distribution.groupby("Customer type")["Transactions"].transform("sum")
payment_distribution["Share within customer type (%)"] = (
    payment_distribution["Transactions"] / payment_distribution["Customer type total"] * 100
)
payment_distribution.to_csv("payment_distribution.csv", index=False, encoding="utf-8-sig")

display(results_df.round(4))
display(contingency)
display(payment_distribution.round(2))

print(f"Records analysed: {len(df)}")
print(f"Alpha: {ALPHA}; Holm correction applied to {len(results_df)} p-values.")
print(f"Significant tests after Holm correction: {int(results_df['Significant_After_Holm'].sum())}/{len(results_df)}")
print("Interpretation: a non-significant result means insufficient evidence of a difference/association, not proof that groups are identical.")
